
# GVH Nested Diagonal Dynamic Bridge 0.3 — PATCH1
## Homogeneous Metricity Gate

**Statut :** patch mathématique local  
**Parent exécuté/audité :** `GVH_Nested_Diagonal_Dynamic_Bridge_0.3(1)`  
**Mission unique :** corriger le mélange entre \(c^2(\mathbf n)\) défini sur la sphère unité et la forme quadratique homogène utilisée pour reconstruire une co-métrique.

Le patch introduit explicitement :

\[
\boxed{
F(\mathbf k)
=
|\mathbf k|^2
c^2\!\left(\hat{\mathbf k}\right),
\qquad
\hat{\mathbf k}
=
\frac{\mathbf k}{|\mathbf k|}.
}
\]

Une branche est métrique au sens spatial recherché seulement si :

\[
\boxed{
F(\mathbf k)
=
k_i C^{ij}k_j
}
\]

pour tout \(\mathbf k\neq0\), avec \(C=C^T>0\).

Ce notebook ne dérive encore aucune branche GVH exacte et ne ferme pas le pont dynamique complet.


In [1]:

from __future__ import annotations

import sys, json
from pathlib import Path

import sympy as sp

PARENT_GNDG03 = {
    "executed_size_bytes": 34025,
    "executed_sha256":
        "ad014f00cd9037f0782a26190957da8f13df3f4f3dad2af6483ca566f174778e",
    "source_exact": True,
    "minimal_dynamic_source_identified": True,
    "g_eff_derived_from_GVH": False,
    "metricity_gate_patch_required": True,
}

GNDG03P1_PROVENANCE_PASS = all([
    PARENT_GNDG03["source_exact"],
    PARENT_GNDG03["minimal_dynamic_source_identified"],
    not PARENT_GNDG03["g_eff_derived_from_GVH"],
    PARENT_GNDG03["metricity_gate_patch_required"],
])

assert GNDG03P1_PROVENANCE_PASS

print("Python =", sys.version.split()[0])
print("SymPy =", sp.__version__)
print("GNDG03P1_PROVENANCE_PASS =", GNDG03P1_PROVENANCE_PASS)


Python = 3.13.15
SymPy = 1.14.0
GNDG03P1_PROVENANCE_PASS = True



# 1. Séparer correctement les deux objets

La vitesse au carré est une fonction de direction :

\[
c^2:S^2\to\mathbb R.
\]

Elle dépend de :

\[
\mathbf n=\frac{\mathbf k}{|\mathbf k|}.
\]

La co-métrique spatiale, elle, produit une forme quadratique homogène :

\[
\mathbf k^TC\mathbf k.
\]

Le bon objet à comparer à une forme quadratique n'est donc pas directement \(c^2(\mathbf n)\), mais :

\[
\boxed{
F(\mathbf k)
=
|\mathbf k|^2c^2(\hat{\mathbf k}).
}
\]

Il vérifie automatiquement :

\[
F(\lambda\mathbf k)=\lambda^2F(\mathbf k)
\]

pour \(\lambda>0\).

Si :

\[
c^2(\mathbf n)=n_iC^{ij}n_j,
\]

alors :

\[
\boxed{
F(\mathbf k)=k_iC^{ij}k_j.
}
\]

C'est cette identité homogène qui doit être testée.


In [2]:

kx, ky, kz, lam = sp.symbols(
    "kx ky kz lam",
    real=True
)

r2 = kx**2 + ky**2 + kz**2

# Generic directional speed represented abstractly through a test function
# is not needed for the symbolic homogeneity check; instead we verify the
# metric case exactly.
c11,c22,c33,c12,c13,c23 = sp.symbols(
    "c11 c22 c33 c12 c13 c23",
    real=True
)

C = sp.Matrix([
    [c11,c12,c13],
    [c12,c22,c23],
    [c13,c23,c33],
])

k = sp.Matrix([kx,ky,kz])

F_metric = sp.expand((k.T*C*k)[0])
F_scaled = sp.expand(
    F_metric.subs({
        kx:lam*kx,
        ky:lam*ky,
        kz:lam*kz,
    })
)

GNDG03P1_HOMOGENEOUS_EXTENSION_DEFINED = (
    sp.simplify(F_scaled-lam**2*F_metric) == 0
)

assert GNDG03P1_HOMOGENEOUS_EXTENSION_DEFINED

print(
    "GNDG03P1_HOMOGENEOUS_EXTENSION_DEFINED =",
    GNDG03P1_HOMOGENEOUS_EXTENSION_DEFINED
)


GNDG03P1_HOMOGENEOUS_EXTENSION_DEFINED = True



# 2. Reconstruction exacte d'une co-métrique

Pour une vraie forme quadratique homogène :

\[
F(\mathbf k)=\mathbf k^TC\mathbf k,
\]

les six évaluations :

\[
e_1,\quad e_2,\quad e_3,\quad
e_1+e_2,\quad
e_1+e_3,\quad
e_2+e_3
\]

suffisent à reconstruire une matrice symétrique \(C\).

En particulier :

\[
C_{11}=F(e_1),
\qquad
C_{22}=F(e_2),
\qquad
C_{33}=F(e_3),
\]

\[
C_{12}
=
\frac{
F(e_1+e_2)-C_{11}-C_{22}
}{2},
\]

et de même pour \(C_{13}\) et \(C_{23}\).

Cette formule est maintenant appliquée uniquement à \(F(\mathbf k)\), jamais directement à des vitesses normalisées \(c^2(\mathbf n)\).


In [3]:

e1 = sp.Matrix([1,0,0])
e2 = sp.Matrix([0,1,0])
e3 = sp.Matrix([0,0,1])

def qform(M,v):
    return sp.expand((v.T*M*v)[0])

def reconstruct_C_from_homogeneous_F(
    F1,F2,F3,F12,F13,F23
):
    C11 = sp.simplify(F1)
    C22 = sp.simplify(F2)
    C33 = sp.simplify(F3)
    C12 = sp.simplify((F12-C11-C22)/2)
    C13 = sp.simplify((F13-C11-C33)/2)
    C23 = sp.simplify((F23-C22-C33)/2)

    return sp.Matrix([
        [C11,C12,C13],
        [C12,C22,C23],
        [C13,C23,C33],
    ])

probes = [
    e1,
    e2,
    e3,
    e1+e2,
    e1+e3,
    e2+e3,
]

C_target = sp.diag(
    sp.Integer(1),
    sp.Integer(1),
    sp.Rational(1,4),
)

F_metric_values = [
    qform(C_target,v)
    for v in probes
]

C_recovered = reconstruct_C_from_homogeneous_F(
    *F_metric_values
)

GNDG03P1_SYNTHETIC_METRIC_RECONSTRUCTION_EXACT = (
    sp.simplify(C_recovered-C_target)
    == sp.zeros(3)
)

assert GNDG03P1_SYNTHETIC_METRIC_RECONSTRUCTION_EXACT

print("C_target =")
sp.pprint(C_target)
print("C_recovered =")
sp.pprint(C_recovered)
print(
    "GNDG03P1_SYNTHETIC_METRIC_RECONSTRUCTION_EXACT =",
    GNDG03P1_SYNTHETIC_METRIC_RECONSTRUCTION_EXACT
)


C_target =
⎡1  0   0 ⎤
⎢         ⎥
⎢0  1   0 ⎥
⎢         ⎥
⎣0  0  1/4⎦
C_recovered =
⎡1  0   0 ⎤
⎢         ⎥
⎢0  1   0 ⎥
⎢         ⎥
⎣0  0  1/4⎦
GNDG03P1_SYNTHETIC_METRIC_RECONSTRUCTION_EXACT = True



# 3. Contre-exemple non métrique corrigé

Reprenons :

\[
\boxed{
c^2(\mathbf n)
=
1+\mu n_x^2n_z^2,
\qquad
\mu=\frac15.
}
\]

Son extension homogène correcte est :

\[
\begin{aligned}
F(\mathbf k)
&=
|\mathbf k|^2
\left[
1+\mu
\frac{k_x^2}{|\mathbf k|^2}
\frac{k_z^2}{|\mathbf k|^2}
\right]\\[1mm]
&=
\boxed{
|\mathbf k|^2
+
\mu
\frac{k_x^2k_z^2}{|\mathbf k|^2}
}.
\end{aligned}
\]

Cette fonction est homogène de degré 2, mais elle n'est pas en général une forme quadratique polynomiale.

On ajuste \(C\) sur les six probes, puis on teste une septième direction indépendante :

\[
\mathbf k_*=(1,1,1).
\]

Un résidu exact non nul rejette la métricité quadratique.


In [4]:

mu = sp.Rational(1,5)

def F_nonmetric(v):
    v = sp.Matrix(v)
    vv = sp.expand((v.T*v)[0])

    if vv == 0:
        raise ValueError("k must be non-zero")

    return sp.simplify(
        vv
        + mu*v[0]**2*v[2]**2/vv
    )

F_nonmetric_values = [
    F_nonmetric(v)
    for v in probes
]

C_fit_nonmetric = reconstruct_C_from_homogeneous_F(
    *F_nonmetric_values
)

k_test = sp.Matrix([1,1,1])

F_true = sp.simplify(
    F_nonmetric(k_test)
)

F_quadratic_fit = sp.simplify(
    qform(C_fit_nonmetric,k_test)
)

F_residual = sp.simplify(
    F_true-F_quadratic_fit
)

c2_residual_unit = sp.simplify(
    F_residual
    / qform(sp.eye(3),k_test)
)

GNDG03P1_NONMETRIC_REJECTION_EXACT = (
    F_residual == -sp.Rational(1,30)
    and c2_residual_unit == -sp.Rational(1,90)
)

assert GNDG03P1_NONMETRIC_REJECTION_EXACT

print("C_fit_nonmetric =")
sp.pprint(C_fit_nonmetric)
print("F_true(k*) =", F_true)
print("F_quadratic_fit(k*) =", F_quadratic_fit)
print("F_residual =", F_residual)
print("normalized c^2 residual =", c2_residual_unit)
print(
    "GNDG03P1_NONMETRIC_REJECTION_EXACT =",
    GNDG03P1_NONMETRIC_REJECTION_EXACT
)


C_fit_nonmetric =
⎡ 1    0  1/20⎤
⎢             ⎥
⎢ 0    1   0  ⎥
⎢             ⎥
⎣1/20  0   1  ⎦
F_true(k*) = 46/15
F_quadratic_fit(k*) = 31/10
F_residual = -1/30
normalized c^2 residual = -1/90
GNDG03P1_NONMETRIC_REJECTION_EXACT = True



# 4. Porte de métricité corrigée

Le test spatial correct devient :

\[
\boxed{
F_a(\mathbf k)
=
|\mathbf k|^2
c_a^2(\hat{\mathbf k})
}
\]

puis :

\[
\boxed{
F_a(\mathbf k)
\stackrel{?}{=}
k_i C_a^{ij}k_j
}
\]

globalement en \(\mathbf k\).

Pour qu'une branche définisse une métrique riemannienne spatiale effective, il faudra simultanément :

1. extraire exactement \(c_a^2(\mathbf n)\) depuis le secteur physique GVH ;
2. construire \(F_a(\mathbf k)\) ;
3. montrer que \(F_a\) est exactement quadratique ;
4. reconstruire \(C_a\) ;
5. établir :
   \[
   C_a=C_a^T;
   \]
6. établir :
   \[
   C_a>0;
   \]
7. définir :
   \[
   g_{{\rm eff},a}=C_a^{-1}.
   \]

Une interpolation numérique ne suffira pas à fermer ce verrou.


In [5]:

GNDG03P1_METRICITY_GATE_FORMULATION_CLEAN = all([
    GNDG03P1_HOMOGENEOUS_EXTENSION_DEFINED,
    GNDG03P1_SYNTHETIC_METRIC_RECONSTRUCTION_EXACT,
    GNDG03P1_NONMETRIC_REJECTION_EXACT,
])

GNDG03P1_SOURCE_IDENTIFICATION_INHERITED = (
    PARENT_GNDG03[
        "minimal_dynamic_source_identified"
    ]
)

GNDG03P1_EXACT_GVH_BRANCH_EXTRACTED = False
GNDG03P1_GEFF_DERIVED_FROM_GVH = False
GNDG03P1_ETA_DYNAMIC_ROLE_ESTABLISHED = False
GNDG03P1_DYNAMIC_GVH_BRIDGE_ESTABLISHED = False

assert GNDG03P1_METRICITY_GATE_FORMULATION_CLEAN
assert GNDG03P1_SOURCE_IDENTIFICATION_INHERITED
assert not GNDG03P1_EXACT_GVH_BRANCH_EXTRACTED
assert not GNDG03P1_GEFF_DERIVED_FROM_GVH
assert not GNDG03P1_DYNAMIC_GVH_BRIDGE_ESTABLISHED

print(
    "GNDG03P1_METRICITY_GATE_FORMULATION_CLEAN =",
    GNDG03P1_METRICITY_GATE_FORMULATION_CLEAN
)
print(
    "GNDG03P1_SOURCE_IDENTIFICATION_INHERITED =",
    GNDG03P1_SOURCE_IDENTIFICATION_INHERITED
)
print(
    "GNDG03P1_GEFF_DERIVED_FROM_GVH =",
    GNDG03P1_GEFF_DERIVED_FROM_GVH
)


GNDG03P1_METRICITY_GATE_FORMULATION_CLEAN = True
GNDG03P1_SOURCE_IDENTIFICATION_INHERITED = True
GNDG03P1_GEFF_DERIVED_FROM_GVH = False



# 5. Verrou supplémentaire pour le futur test complet

La formulation spatiale :

\[
\omega^2=C^{ij}k_i k_j
\]

suppose une forme particulière du facteur caractéristique.

Pour un test réellement covariant du cône de propagation, l'objet plus général est le covecteur :

\[
p_\mu=(\omega,k_i).
\]

Une métrique effective complète correspondrait à un facteur homogène quadratique :

\[
\boxed{
G_{\rm eff}^{\mu\nu}p_\mu p_\nu=0.
}
\]

Cela peut contenir :

\[
\omega^2,
\qquad
\omega k_i,
\qquad
k_i k_j.
\]

Par conséquent, lors de l'extraction des vraies branches GVH, il faudra d'abord déterminer si les termes mixtes \(\omega k_i\) sont absents ou éliminables dans le secteur considéré.

Ce patch ne ferme pas ce futur verrou ; il l'explicite pour éviter une promotion prématurée d'une simple co-métrique spatiale en métrique effective complète.


In [6]:

GNDG03P1_FULL_COVECTOR_METRICITY_TEST_REQUIRED = True
GNDG03P1_MIXED_OMEGA_K_TERMS_CERTIFIED_ABSENT = False

GNDG03P1_NEXT_LOCK = (
    "EXTRACT_EXACT_GLOBAL_PHYSICAL_BRANCH_AND_TEST_"
    "HOMOGENEOUS_QUADRATIC_METRICITY"
)

FOUR_LEVEL_PROTOCOL_PASS = True
UNIVERSAL_THEORY_SELECTED_SI_SCALE_RANK = 0

verdict = {
    "notebook":
        "GVH_Nested_Diagonal_Dynamic_Bridge_0.3_PATCH1_Homogeneous_Metricity_Gate",
    "parent":
        PARENT_GNDG03,
    "patch":{
        "homogeneous_extension_defined":
            bool(GNDG03P1_HOMOGENEOUS_EXTENSION_DEFINED),
        "synthetic_metric_reconstruction_exact":
            bool(GNDG03P1_SYNTHETIC_METRIC_RECONSTRUCTION_EXACT),
        "nonmetric_rejection_exact":
            bool(GNDG03P1_NONMETRIC_REJECTION_EXACT),
        "metricity_gate_formulation_clean":
            bool(GNDG03P1_METRICITY_GATE_FORMULATION_CLEAN),
        "correct_nonmetric_F_residual":
            str(F_residual),
        "correct_normalized_speed_residual":
            str(c2_residual_unit),
    },
    "inherited":{
        "minimal_dynamic_source_identified":
            bool(GNDG03P1_SOURCE_IDENTIFICATION_INHERITED),
    },
    "open_locks":{
        "exact_GVH_branch_extracted":
            False,
        "g_eff_derived_from_GVH":
            False,
        "eta_dynamic_role_established":
            False,
        "dynamic_GVH_bridge_established":
            False,
        "full_covector_metricity_test_required":
            True,
        "mixed_omega_k_terms_certified_absent":
            False,
    },
    "next_lock":
        GNDG03P1_NEXT_LOCK,
    "protocol":{
        "four_level_protocol_pass":
            True,
        "universal_theory_selected_SI_scale_rank":
            0,
    },
    "status":
        "PASS_METRICITY_GATE_PATCHED_EXACT_BRANCH_EXTRACTION_OPEN",
}

export_dir = Path(
    "/mnt/data/gvh_nested_dynamic_bridge_0_3_patch1_exports"
)
export_dir.mkdir(
    parents=True,
    exist_ok=True
)

verdict_path = (
    export_dir
    / "gvh_dynamic_bridge_0.3_PATCH1_verdict.json"
)

verdict_path.write_text(
    json.dumps(
        verdict,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("STATUS =", verdict["status"])
print(
    "METRICITY_GATE_FORMULATION_CLEAN =",
    verdict["patch"][
        "metricity_gate_formulation_clean"
    ]
)
print(
    "CORRECT_F_RESIDUAL =",
    verdict["patch"][
        "correct_nonmetric_F_residual"
    ]
)
print(
    "G_EFF_DERIVED_FROM_GVH =",
    verdict["open_locks"][
        "g_eff_derived_from_GVH"
    ]
)
print(
    "NEXT_LOCK =",
    verdict["next_lock"]
)
print("verdict JSON =", verdict_path)


STATUS = PASS_METRICITY_GATE_PATCHED_EXACT_BRANCH_EXTRACTION_OPEN
METRICITY_GATE_FORMULATION_CLEAN = True
CORRECT_F_RESIDUAL = -1/30
G_EFF_DERIVED_FROM_GVH = False
NEXT_LOCK = EXTRACT_EXACT_GLOBAL_PHYSICAL_BRANCH_AND_TEST_HOMOGENEOUS_QUADRATIC_METRICITY
verdict JSON = /mnt/data/gvh_nested_dynamic_bridge_0_3_patch1_exports/gvh_dynamic_bridge_0.3_PATCH1_verdict.json
